# Session 7: Introduction to LangGraph
### Agentic AI Nano Bootcamp | Day 2, Session 7

---

## Learning Objectives

By the end of this session, you will be able to:
- Explain why graph-based agent frameworks outperform linear chains for complex workflows
- Define the core LangGraph concepts: StateGraph, nodes, edges, and conditional routing
- Build a single stateful agent using LangGraph
- Implement conditional branching based on agent output
- Construct a multi-step Q&A workflow connecting LangChain tools to a LangGraph agent

## Session Outline

1. Why Graph-Based Agent Frameworks?
2. LangGraph Core Concepts
3. LangChain vs LangGraph — When to Use Each
4. State Management in LangGraph
5. Lab 1: First LangGraph Agent (Single Node)
6. Lab 2: Multi-Node Agent with Conditional Routing
7. Lab 3: Multi-Step Q&A Workflow with LangChain + LangGraph

---

In [ ]:
import subprocess, os, json, operator, datetime
subprocess.run(['pip', 'install', 'langgraph', 'langchain', 'langchain-openai',
                'langchain-community', 'openai', 'python-dotenv', '-q'], capture_output=True)

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
plain_client = OpenAI()
print("Dependencies ready.")

## 1. Why Graph-Based Agent Frameworks?

The agents in Sessions 5 and 6 used an explicit Python loop. That works for simple agents, but becomes unmanageable as complexity grows:

| Challenge | Loop-based Agent | Graph-based Agent |
|---|---|---|
| Conditional routing | Nested if/else in Python | Declared edges — readable and maintainable |
| State passing | Manual dict management | Typed state schema, automatic propagation |
| Cycles and loops | Risk of infinite recursion | Controlled via edges and guards |
| Multi-agent coordination | Custom message passing | Nodes compose naturally |
| Visualisation | Not possible | Built-in graph drawing |
| Checkpointing | Custom implementation | Built-in persistence and replay |

### The Fundamental Insight

An agent workflow is a **directed graph** where:
- Nodes are processing steps (LLM calls, tool calls, transforms)
- Edges define what happens next
- State flows through the graph and is updated at each node

LangGraph makes this structure explicit rather than implicit.

## 2. LangGraph Core Concepts

### StateGraph

The `StateGraph` is the container for the entire workflow. It holds the state schema, nodes, and edges.

```python
from langgraph.graph import StateGraph, END

graph = StateGraph(MyState)   # MyState is a TypedDict
```

### State

State is a typed dictionary that is shared across all nodes. Each node receives the current state and returns an update.

```python
from typing import TypedDict, Annotated

class AgentState(TypedDict):
    messages:  Annotated[list, operator.add]   # add appends new messages
    question:  str
    answer:    str
    step_count: int
```

The `Annotated[list, operator.add]` pattern means updates to `messages` are **appended** rather than replaced.

### Nodes

A node is any callable `(state) -> state_update`. It receives the full state and returns only the fields it wants to update.

```python
def my_node(state: AgentState) -> dict:
    # process state
    return {"answer": "updated value"}    # only modified fields

graph.add_node("my_node", my_node)
```

### Edges

```python
graph.add_edge("node_a", "node_b")         # unconditional: always go to node_b
graph.add_edge("node_b", END)              # terminate

graph.add_conditional_edges(               # conditional: route based on function
    "node_a",
    routing_function,                       # returns a string (node name)
    {"path_1": "node_b", "path_2": "node_c"}
)
```

### Entry Point and Compilation

```python
graph.set_entry_point("first_node")
app = graph.compile()                      # creates runnable
result = app.invoke({"question": "..."})
```

## 3. LangChain vs LangGraph — When to Use Each

| Scenario | LangChain | LangGraph |
|---|---|---|
| Simple RAG pipeline | Ideal — LCEL chains are clean and fast | Overkill |
| Fixed sequence with no branching | Ideal | Overkill |
| Agent with conditional routing | Awkward — nested if/else | Ideal |
| Stateful multi-turn conversations | Possible | Natural — state is first-class |
| Multi-agent coordination | Requires custom code | Built-in via subgraph composition |
| Checkpointing and replay | Not supported | Built-in |
| Visualising the workflow | Not supported | Built-in |

**Rule of thumb**: use LangChain (LCEL) for pipelines, use LangGraph for agents with decision logic.

The two are designed to work together — LangChain components (retrievers, LLMs, tools) plug directly into LangGraph nodes.

## Lab 1: First LangGraph Agent (Single Node)

Start simple: a single-node graph that takes a question and returns an answer. This establishes the pattern before adding complexity.

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, List
import operator

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# 1. Define the state schema
class SimpleState(TypedDict):
    messages: Annotated[list, operator.add]
    final_answer: str

# 2. Define the node function
def answer_node(state: SimpleState) -> dict:
    """
    The only node: takes incoming messages, calls the LLM, returns the response.
    """
    messages = state["messages"]
    response = llm.invoke(messages)
    return {
        "messages":     [response],
        "final_answer": response.content
    }

# 3. Build the graph
builder = StateGraph(SimpleState)
builder.add_node("answer", answer_node)
builder.set_entry_point("answer")
builder.add_edge("answer", END)

simple_graph = builder.compile()

print("Graph compiled. Nodes:", list(simple_graph.get_graph().nodes.keys()))

# 4. Invoke
system = SystemMessage(content="You are a concise telecom domain expert.")

test_questions = [
    "What is the difference between FTTH and FTTB?",
    "Name three key metrics for measuring mobile network quality.",
]

for q in test_questions:
    result = simple_graph.invoke({
        "messages": [system, HumanMessage(content=q)],
        "final_answer": ""
    })
    print(f"\nQ: {q}")
    print(f"A: {result['final_answer']}")

## Lab 2: Multi-Node Agent with Conditional Routing

We now build a telecom support agent with three nodes and conditional routing:

```
START
  |
  v
[classify]  --(connectivity)--> [technical_node]  --> END
            --(billing)-------> [billing_node]     --> END
            --(escalate)------> [escalate_node]    --> END
```

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated
import operator

class SupportState(TypedDict):
    customer_query:  str
    category:        str
    messages:        Annotated[list, operator.add]
    final_response:  str
    escalated:       bool

# --- Node: Classifier ---
def classify_node(state: SupportState) -> dict:
    """
    Classify the customer query into one of: connectivity, billing, escalate.
    """
    prompt = f"""
Classify this customer support query into EXACTLY one category.
Return only the category word — no explanation, no punctuation.
Categories:
  connectivity — internet, speed, outage, router, WiFi, network
  billing      — invoice, charge, payment, refund, plan, upgrade
  escalate     — legal threat, extreme distress, regulatory complaint, repeat failure

Query: {state['customer_query']}
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    category = response.content.strip().lower()
    if category not in ("connectivity", "billing", "escalate"):
        category = "connectivity"  # default
    print(f"  [classify_node] Category determined: {category}")
    return {
        "category": category,
        "messages": [HumanMessage(content=state["customer_query"])]
    }

# --- Node: Connectivity specialist ---
def technical_node(state: SupportState) -> dict:
    """
    Handle connectivity and technical queries with diagnostic guidance.
    """
    system = SystemMessage(content=(
        "You are a Level-2 network support engineer. "
        "Provide structured technical troubleshooting steps. "
        "Format: empathy sentence, 3 numbered steps, escalation path if unresolved. "
        "Maximum 120 words."
    ))
    response = llm.invoke([system] + state["messages"])
    print(f"  [technical_node] Response generated.")
    return {"final_response": response.content, "messages": [response], "escalated": False}

# --- Node: Billing specialist ---
def billing_node(state: SupportState) -> dict:
    """
    Handle billing and account queries.
    """
    system = SystemMessage(content=(
        "You are a billing support specialist. "
        "Address billing concerns with empathy. "
        "Do not confirm errors or promise refunds without investigation. "
        "Offer to review within 24 hours. Maximum 100 words."
    ))
    response = llm.invoke([system] + state["messages"])
    print(f"  [billing_node] Response generated.")
    return {"final_response": response.content, "messages": [response], "escalated": False}

# --- Node: Escalation handler ---
def escalate_node(state: SupportState) -> dict:
    """
    Handle high-severity or legal escalation cases.
    """
    system = SystemMessage(content=(
        "You are a senior customer relationship manager handling an escalation. "
        "Acknowledge the severity, assure personal attention from a senior team, "
        "and provide a direct callback commitment. Formal, empathetic tone. 80 words max."
    ))
    response = llm.invoke([system] + state["messages"])
    print(f"  [escalate_node] Escalation response generated.")
    return {"final_response": response.content, "messages": [response], "escalated": True}

# --- Routing function ---
def route_by_category(state: SupportState) -> str:
    """Return the name of the next node based on the classified category."""
    routing_map = {
        "connectivity": "technical",
        "billing":      "billing",
        "escalate":     "escalate",
    }
    return routing_map.get(state["category"], "technical")

# --- Build the graph ---
builder = StateGraph(SupportState)
builder.add_node("classify",  classify_node)
builder.add_node("technical", technical_node)
builder.add_node("billing",   billing_node)
builder.add_node("escalate",  escalate_node)

builder.set_entry_point("classify")
builder.add_conditional_edges(
    "classify",
    route_by_category,
    {"technical": "technical", "billing": "billing", "escalate": "escalate"}
)
builder.add_edge("technical", END)
builder.add_edge("billing",   END)
builder.add_edge("escalate",  END)

support_graph = builder.compile()
print("Support graph compiled.")
print("Nodes:", list(support_graph.get_graph().nodes.keys()))

In [ ]:
# Test the multi-node agent with three different query types

test_cases = [
    "My internet has been down since this morning. Router shows solid green but no connectivity.",
    "I was charged twice for my monthly plan renewal and I want a full refund immediately.",
    "I am contacting TRAI. Your service has failed for the third time this month and I am recording this conversation as evidence.",
]

initial_state_template = {
    "messages":       [],
    "category":       "",
    "final_response": "",
    "escalated":      False
}

for query in test_cases:
    print("\n" + "=" * 65)
    print(f"Customer: {query}")
    print("=" * 65)

    result = support_graph.invoke({
        **initial_state_template,
        "customer_query": query
    })

    print(f"\n  Category:  {result['category']}")
    print(f"  Escalated: {result['escalated']}")
    print(f"\n  Response:")
    print(result['final_response'])

## Lab 3: Multi-Step Q&A Workflow with LangChain + LangGraph

We combine:
- A **LangChain retriever** (vector store from Session 4)
- A **LangGraph agent** with retrieve, generate, and critique nodes
- **Conditional routing**: if the generated answer fails a quality check, loop back to retrieval

```
START
  |
  v
[retrieve] --> [generate] --> [critique]
                    ^               |
                    |-- retry ------+---- pass --> END
```

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated, List
import operator, json, os

# Build a small in-memory telecom knowledge base for this lab
TELECOM_DOCS = [
    "5G uses millimeter wave frequencies above 24 GHz for ultra-high speeds in dense areas, mid-band 1-6 GHz for balanced coverage, and low-band below 1 GHz for wide area coverage.",
    "Network latency in 5G targets sub-1ms for URLLC (Ultra-Reliable Low Latency Communications) applications such as autonomous vehicles and remote surgery.",
    "Churn rate is the percentage of subscribers who cancel their service in a given period. Industry average for postpaid mobile is 1.5-2.5% monthly.",
    "A Base Transceiver Station (BTS) connects mobile devices to the core network via radio frequency transmission. One BTS can serve hundreds to thousands of users simultaneously.",
    "FTTH (Fiber to the Home) delivers fiber optic cable directly to the subscriber premises, offering symmetric speeds up to 10 Gbps and latency under 5ms.",
    "Network slicing in 5G allows a single physical network to be divided into multiple virtual networks each optimised for specific use cases such as IoT, video streaming, or enterprise VPN.",
    "SLA (Service Level Agreement) in telecom typically guarantees 99.9% uptime, mean time to repair under 4 hours for P1 incidents, and packet loss below 0.1%.",
    "LTE (Long Term Evolution) is the 4G standard offering peak download speeds of 150 Mbps on Category 4 devices and up to 1 Gbps on LTE-Advanced Pro.",
    "VoLTE (Voice over LTE) carries voice calls over the 4G LTE data network rather than the legacy 2G/3G circuit-switched network, improving call quality and setup time.",
    "Customer Lifetime Value (CLV) in telecom is calculated as average monthly revenue multiplied by average subscriber tenure in months minus acquisition cost.",
]

embeddings   = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore  = Chroma.from_texts(texts=TELECOM_DOCS, embedding=embeddings)
retriever    = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Knowledge base built: {len(TELECOM_DOCS)} documents")

# --- State schema ---
class QAState(TypedDict):
    question:       str
    retrieved_docs: List[str]
    draft_answer:   str
    final_answer:   str
    quality_score:  int
    retry_count:    int
    messages:       Annotated[list, operator.add]

# --- Node: Retrieve ---
def retrieve_node(state: QAState) -> dict:
    """
    Retrieve relevant documents from the knowledge base.
    On retry, the question is expanded to find better context.
    """
    question = state["question"]
    retry    = state.get("retry_count", 0)

    if retry > 0:
        # Expand the query on retry
        expand_prompt = f"Rephrase this question using more specific telecom terminology to improve search results: {question}"
        expanded = llm.invoke([HumanMessage(content=expand_prompt)]).content
        search_query = expanded
        print(f"  [retrieve_node] Retry {retry} — expanded query: {expanded[:60]}...")
    else:
        search_query = question
        print(f"  [retrieve_node] Retrieving for: {question[:60]}")

    docs = retriever.invoke(search_query)
    doc_texts = [d.page_content for d in docs]
    print(f"  [retrieve_node] Retrieved {len(doc_texts)} documents")
    return {"retrieved_docs": doc_texts}

# --- Node: Generate ---
def generate_node(state: QAState) -> dict:
    """
    Generate an answer from the retrieved documents.
    """
    context = "\n".join(f"[{i+1}] {d}" for i, d in enumerate(state["retrieved_docs"]))
    prompt = f"""
Answer the question using only the provided context.
Cite the source numbers in your answer (e.g. "according to [1]").
If the context does not contain enough information, say so explicitly.

Context:
{context}

Question: {state['question']}
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [generate_node] Answer generated ({len(response.content.split())} words)")
    return {"draft_answer": response.content, "messages": [response]}

# --- Node: Critique ---
def critique_node(state: QAState) -> dict:
    """
    Score the draft answer on completeness and grounding.
    Returns a quality score 1-10.
    """
    prompt = f"""
Evaluate this answer to the given question.
Score from 1 (poor) to 10 (excellent) based on:
- Does it directly answer the question?
- Is it grounded in the provided context (no hallucination)?
- Is it specific rather than vague?

Question: {state['question']}
Answer:   {state['draft_answer']}

Return ONLY a JSON object: {{"score": <1-10>, "reason": "<one sentence>"}}
"""
    raw = llm.invoke(
        [HumanMessage(content=prompt)],
    ).content
    try:
        # strip possible markdown fences
        clean = raw.strip().strip('`').replace('json','').strip()
        parsed = json.loads(clean)
        score  = int(parsed.get("score", 5))
        reason = parsed.get("reason", "")
    except Exception:
        score  = 5
        reason = "Could not parse critique"

    print(f"  [critique_node] Score: {score}/10 — {reason}")
    return {"quality_score": score}

# --- Node: Finalise ---
def finalise_node(state: QAState) -> dict:
    """Accept the draft answer as the final answer."""
    print(f"  [finalise_node] Answer accepted with score {state['quality_score']}/10")
    return {"final_answer": state["draft_answer"]}

# --- Node: Retry ---
def retry_node(state: QAState) -> dict:
    """Increment retry counter and feed back to retrieve."""
    count = state.get("retry_count", 0) + 1
    print(f"  [retry_node] Quality below threshold. Retry count: {count}")
    return {"retry_count": count, "draft_answer": ""}

# --- Routing function ---
def route_after_critique(state: QAState) -> str:
    score      = state.get("quality_score", 0)
    retry_count = state.get("retry_count", 0)
    if score >= 7 or retry_count >= 2:
        return "finalise"
    return "retry"

# --- Build graph ---
builder = StateGraph(QAState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("critique", critique_node)
builder.add_node("finalise", finalise_node)
builder.add_node("retry",    retry_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "critique")
builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"finalise": "finalise", "retry": "retry"}
)
builder.add_edge("retry",    "retrieve")
builder.add_edge("finalise", END)

qa_graph = builder.compile()
print("\nRAG-with-critique graph compiled.")
print("Nodes:", list(qa_graph.get_graph().nodes.keys()))

In [ ]:
# Run the multi-node RAG workflow

questions = [
    "What are the three frequency bands used in 5G and what are the trade-offs?",
    "How is Customer Lifetime Value calculated in telecom?",
    "What latency does 5G target and for which use cases?",
]

for q in questions:
    print("\n" + "=" * 65)
    print(f"Question: {q}")
    print("=" * 65)

    result = qa_graph.invoke({
        "question":       q,
        "retrieved_docs": [],
        "draft_answer":   "",
        "final_answer":   "",
        "quality_score":  0,
        "retry_count":    0,
        "messages":       []
    })

    print(f"\n  Final answer (quality {result['quality_score']}/10):")
    print(result["final_answer"])

In [ ]:
# Visualise the graph structure
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('LangGraph Workflows Built in This Session', fontsize=13, fontweight='bold')

def draw_graph(ax, title, nodes_list, edges_list, conditional_edges=None):
    ax.set_xlim(0, 8)
    ax.set_ylim(0, len(nodes_list) + 1)
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold')
    node_positions = {}
    for i, (name, color) in enumerate(nodes_list):
        y = len(nodes_list) - i
        node_positions[name] = (4.0, y)
        ax.add_patch(mpatches.FancyBboxPatch(
            (2.5, y - 0.3), 3.0, 0.6, boxstyle='round,pad=0.1', fc=color, ec='#444', lw=1))
        ax.text(4.0, y, name, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    for src, dst in edges_list:
        x1, y1 = node_positions.get(src, (4, 0))
        x2, y2 = node_positions.get(dst, (4, 0))
        ax.annotate('', xy=(x2, y2 + 0.3), xytext=(x1, y1 - 0.3),
                    arrowprops=dict(arrowstyle='->', color='#444', lw=1.5))
    if conditional_edges:
        for src, branches in conditional_edges:
            x1, y1 = node_positions.get(src, (4, 0))
            for dst, label in branches:
                x2, y2 = node_positions.get(dst, (4, 0))
                ax.annotate('', xy=(x2, y2 + 0.3), xytext=(x1, y1 - 0.3),
                            arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.5, linestyle='dashed'))
                mx, my = (x1 + x2) / 2, (y1 + y2) / 2
                ax.text(mx + 0.5, my, label, fontsize=7.5, color='#C0392B')

# Support agent graph
draw_graph(
    axes[0], 'Support Agent (Lab 2)',
    [("classify", "#1F4E79"), ("technical", "#1B5E3B"),
     ("billing", "#7B3F00"), ("escalate", "#C0392B")],
    [],
    conditional_edges=[
        ("classify", [("technical", "connectivity"), ("billing", "billing"), ("escalate", "escalate")])
    ]
)

# RAG critique graph
draw_graph(
    axes[1], 'RAG with Critique (Lab 3)',
    [("retrieve", "#1F4E79"), ("generate", "#2E75B6"),
     ("critique", "#6C3483"), ("retry", "#C0392B"), ("finalise", "#1B5E3B")],
    [("retrieve", "generate"), ("generate", "critique"), ("retry", "retrieve")],
    conditional_edges=[
        ("critique", [("finalise", "pass (>=7)"), ("retry", "retry (<7)")])
    ]
)

plt.tight_layout()
plt.savefig('langgraph_workflows.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graph diagram saved.")

In [ ]:
# Exercise: Add a new node to the support graph
# Implement an 'account' node for queries about account management.
# Update the classifier to also detect the 'account' category.
# Route the new category to your new node.

# Starter code:

def account_node(state: SupportState) -> dict:
    """
    Handle account management queries: password reset, plan changes, SIM swap.
    Implement this node.
    """
    # TODO: implement
    return {"final_response": "Account node not yet implemented.", "escalated": False}

print("Starter code provided. Implement account_node and integrate it into the support_graph.")
print("Steps:")
print("  1. Implement account_node using an appropriate system prompt")
print("  2. Add 'account' as a category in classify_node")
print("  3. Add the node to the builder and add an edge to END")
print("  4. Update route_by_category to include 'account'")
print("  5. Test with: 'I need to change my registered email address'")

## Session Summary

| Concept | Key Takeaway |
|---|---|
| StateGraph | Container for nodes, edges, and typed shared state |
| Nodes | Any callable that takes state and returns a partial update |
| Edges | Declare what happens next — unconditional or conditional |
| Conditional routing | A function that returns a node name based on state |
| State annotation | `Annotated[list, operator.add]` appends; plain assignment replaces |
| Retry loop | Conditional edge pointing back to an earlier node enables quality-driven iteration |
| LangChain + LangGraph | Retrievers and LLM chains plug directly into LangGraph nodes |

## What Is Next

Session 8 is the **capstone case study**: evaluate a lightweight GenAI model, then build and demonstrate a complete telecom task automation bot using LangGraph. Teams will present working agents.

---
*Agentic AI Nano Bootcamp | Day 2, Session 7*